# 01 · The frames — a typed storage contract for `(time, unit)` arrays

**What this teaches:** what a `views_frames` frame *is* and what it can do, as a storage device — the immutable, numpy-only array+identifier value objects at the root of the VIEWS dependency graph, and why they exist (to replace the list-in-cell `object`-dtype DataFrame that caused the ~33× memory blow-up / report-stage OOM, README §7).

**Audience:** a consumer-repo author about to migrate a DataFrame-based path onto frames.

> **🚧 ROADMAP — draft, not yet built.** This notebook currently holds only the *plan* (markdown). The section outline below is what we align on **before** the first content pass; code cells get filled in afterwards. Edit these markdown cells freely — this is the working document for the notebook's design.

## 📋 Roadmap (section-by-section)

**1. Motivation — the problem the frame solves.** The list-in-cell DataFrame (a cell holding a Python list of S draws): show the ~33× blow-up conceptually; contrast with a dense `float32 (N, S)` array + integer identifiers. *One short before/after.*

**2. `SpatioTemporalIndex`** — the shared `{time, unit, level}` identifier object; `SpatialLevel` as label-only vocabulary (cm/pgm); same-level numpy alignment. Build one; show `.time` / `.unit` / `.level`.

**3. The three frames as *siblings* (no base class — ADR-011).** Build each from synthetic arrays: `PredictionFrame (N, S)` (a posterior sample per row), `TargetFrame (N, 1)` (a degenerate 1-sample frame), `FeatureFrame (N, F, S)` (+ `from_2d`). Show `.values`, `.identifiers`, `.n_rows`, `.sample_count`, `.is_sample`. **Teach the family, not three separate things** — why siblings sharing one index beats a `_BaseFrame` god-class.

**4. Immutability & structural sharing (C-07).** Operations return *new* frames; structural ops share the buffer (zero-copy), only reductions allocate. Demo `select`, `reindex`, `with_metadata`; show the input is never mutated.

**5. Cross-level alignment (ADR-014).** `cross_level_align` / `cross_level_align_arrays` — remap pgm rows to their cm unit via an **injected** `(time, unit) → country` mapping. Emphasise: the leaf owns the *operation*, the caller injects the *mapping*; geography is never embedded or fetched.

**6. Serialization round-trips.** `frame.save()/load()` (npz, mmap-capable) and the columnar `io.arrow` (parquet) codec; prove `frame in == frame out`. *This is the natural place to surface the `to_parquet` ergonomic gap (register D-11) — show the current `io.arrow.save(**state)` destructuring and note the open question.*

**7. Fail-loud construction (ADR-008/009).** A few invariant violations that `raise` at build time (wrong dtype, mismatched lengths, missing trailing axis) — the structural guarantee a downstream repo can rely on.

**8. Metadata & provenance.** `FrameMetadata` (`run_id` / `data_version`), `to_dict`/`from_dict`; the published `assert_frame_envelope` conformance check as a coda.

## 🔬 Additions from the expert-method-review panel (2026-06-26)

*Folded in from the data-scientist/analyst panel (Gelman · Gneiting · Hyndman · Wilke/Kay · VIEWS-FAO). Tracked as register C-59/C-60/C-61.*

- **A. Ground-truth-carrying synthetic data (supports notebooks 02–03).** The synthetic generators used across all three notebooks should expose the **known latent truth** of each distribution (true mode / mean / quantiles / country total), so 02 can check *coverage* and *recovery* and 03 can ask *does reconciliation help*. Build this into the shared data helper here. *(Gelman — posterior-predictive / recovery checks.)*

## 🎨 Source material — self-contained

Built fresh from small synthetic arrays; depends on no file outside this repo. Optional tone/structure reference: `views-faoapi/notebooks/quickstart_forecast.ipynb` (in-platform).

## ❓ Open questions / decisions

1. **All three frames in one notebook** (my recommendation — teaches the family) vs one section each. Confirm.
2. How much to show the **conformance suite** (`assert_frame_contract` / `assert_frame_envelope`) — full section, or a one-cell coda?
3. Do we show the **arrow/parquet** round-trip here, or defer all serialization to its own mini-notebook? (Leaning: show it here, briefly.)
4. Realism of the synthetic `(time, unit)` ids — toy integers, or fake-but-plausible priogrid/country ids for continuity with notebook 03?

## ✅ Conventions for these notebooks

- **Public, frozen API only.** Everything shown uses the published `views_frames` / `views_frames_summarize` / `views_frames_reconcile` surface — so the notebook doubles as a living contract demo. If a cell *wants* a convenience that doesn't exist (e.g. `frame.to_parquet`), that's a **demand signal** to record, not a reason to reach into the leaf (see register D-11).
- **Synthetic data only.** No `viewser`, no domain fetches, no `views_*` consumer imports — every array is generated in-notebook with a fixed seed. Zero domain knowledge (ADR-001).
- **Un-gated dev artifact.** Lives in `notebooks/` (like `research/` and `scripts/`), outside `src/`; the package never imports it, so the import-DAG and the frozen core are untouched. Plotting/jupyter deps are an **optional extra**, never runtime deps.
- **Reproducible + light.** Fixed seeds; small sample sizes that render fast; no heavyweight state. A reader should be able to *Run All* in well under a minute.